# Combine LLM Classifiers Output

- After running `llm-classifiers.py` per LLM, each output is saved in `data/classification_results/[dataset_folder_name]/seed[...]/in_domain/`
    1. `checkpoint_[model_name].csv`: llm label (0: non-prediction, 1: prediction) per sample/sentence
    2. `metrics_summary_[model_name].csv`: test accuracy, precision, recall, etc

- Tasks: 
    1. Combine the `checkpoint_[model_name].csv` by having all models into joint csv.
    2. Average and standard deviation on `metrics_summary_[model_name].csv` across all seeds.

In [1]:
import os
import sys

import pandas as pd

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_columns', 40)
# pd.set_option('display.max_rows', None)

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
dataset_folder_path = os.path.join(base_data_path, 'classification_results', 'synthetic-fpb-chronicle2050-yt-news-timebank_2026-04-18')

In [4]:
dir_files = os.listdir(dataset_folder_path)
dir_files

['seed3', 'averaged', 'seed7', 'seed33']

In [8]:
models_dfs = []
metrics_dfs = []

for dir_file in dir_files:
    if "seed" in dir_file:
        # print(True, dir_file)
        seed_path = os.path.join(dataset_folder_path, dir_file, 'in_domain')
        # print(True, dir_file, seed_path)
        seed_path_files = os.listdir(seed_path)
        # print(seed_path_files)
        for seed_path_file in seed_path_files:
            if "checkpoint" in seed_path_file:
                print(f"Checking if 'checkpoint' is in \n\t{seed_path_file}")
                checkpoint_path = os.path.join(seed_path, seed_path_file)
                # print(f"\t{seed_path_file}, {checkpoint_path}")
                df = DataProcessing.load_from_file(
                    path=checkpoint_path
                )
                models_dfs.append(df)
            # print("checking:", seed_path_file)
            if "metrics" in seed_path_file and "ml" not in seed_path_file:
                print(f"Checking if 'metrics' is in \n\t{seed_path_file}")
                metrics_path = os.path.join(seed_path, seed_path_file)
                # print(True, seed_path_file, metrics_path)
                df = DataProcessing.load_from_file(
                    path=metrics_path
                )
                metrics_dfs.append(df)

models_df = DataProcessing.concat_dfs(models_dfs)
metrics_df = DataProcessing.concat_dfs(metrics_dfs)

Checking if 'checkpoint' is in 
	checkpoint_gemma-3-27b-it.csv
Checking if 'metrics' is in 
	metrics_summary_gemma-3-27b-it.csv
Checking if 'checkpoint' is in 
	checkpoint_mistral-small-3.1.csv
Checking if 'checkpoint' is in 
	checkpoint_gpt-oss-120b.csv
Checking if 'checkpoint' is in 
	checkpoint_llama-3.1-70b-instruct.csv
Checking if 'metrics' is in 
	metrics_summary_mistral-small-3.1.csv
Checking if 'checkpoint' is in 
	checkpoint_openai_gpt-oss-120b.csv
Checking if 'metrics' is in 
	metrics_summary_gpt-oss-120b.csv
Checking if 'checkpoint' is in 
	checkpoint_gemma-3-27b-it.csv
Checking if 'metrics' is in 
	metrics_summary_gemma-3-27b-it.csv
Checking if 'checkpoint' is in 
	checkpoint_mistral-small-3.1.csv
Checking if 'checkpoint' is in 
	checkpoint_gpt-oss-120b.csv
Checking if 'metrics' is in 
	metrics_summary_mistral-small-3.1.csv
Checking if 'metrics' is in 
	metrics_summary_gpt-oss-120b.csv
Checking if 'checkpoint' is in 
	checkpoint_gemma-3-27b-it.csv
Checking if 'metrics' is i

In [11]:
models_df

,original_index,text,raw_response,llm_label,llm_name
0,0,"Q: In 2020, I lived in a three-bedroom apartment with two roommates.","{""predicted_sentence_label"": 0}",0,gemma-3-27b-it
1,1,"Simultaneously , Alma Media has purchased a 35 % share of Arena Interactive , a subsidiary of Arena Partners with a focus on mobile solutions development .","{""predicted_sentence_label"": 0}",0,gemma-3-27b-it
2,2,"""At some point, it will become non-economical for one company.","{""predicted_sentence_label"": 1}",1,gemma-3-27b-it
3,3,"In addition, Upjohn is offering a one-time retirement bonus equal to six months of base pay.","{""predicted_sentence_label"": 0}",0,gemma-3-27b-it
4,4,Earnings per share were higher at 0.48 against 0.37 a year before and ahead of market consensus of 0.40 eur .,"{""predicted_sentence_label"": 0}",0,gemma-3-27b-it
...,...,...,...,...,...
21775,2415,"Computer experts familiar with the flaws, found in Intel's 80486 chip, say the defects do n't affect the average user and are likely to be cleared up before most computers using the chip as their ""brains"" appear on the market sometime next year.","{""predicted_sentence_label"": 1}",1,gpt-oss-120b
21776,2416,"With Why Is We Americans?, a documentary about the impact the poet and radical Amiri Baraka and his descendants have had on the city of Newark, the directors Udi Aloni and Ayana Stafford-Morris attempt a different approach.","{""predicted_sentence_label"": 0}",0,gpt-oss-120b
21777,2417,"On February 22, 2028, the Urban Institute speculated the budget allocation for social services at city councils changed.","{""predicted_sentence_label"": 1}",1,gpt-oss-120b
21778,2418,"At GMAC, net dropped 3.1% to $234.5 million from $241.9 million.","{""predicted_sentence_label"": 0}",0,gpt-oss-120b


In [12]:
metrics_df

,model,train_accuracy,val_accuracy,test_accuracy,precision_class_0,precision_class_1,recall_class_0,recall_class_1,f1_class_0,f1_class_1,tn,fp,fn,tp,roc_auc,pr_auc
0,gemma-3-27b-it,NaN,NaN,0.861983,0.962139,0.641534,0.855235,0.885036,0.905543,0.743865,1601,271,63,485,0.870136,0.593814
1,mistral-small-3.1,NaN,NaN,0.871074,0.945714,0.676119,0.884081,0.826642,0.913860,0.743842,1655,217,95,453,0.855362,0.598165
2,gpt-oss-120b,NaN,NaN,0.885950,0.947811,0.713166,0.902244,0.830292,0.924466,0.767285,1689,183,93,455,0.866268,0.630566
3,gemma-3-27b-it,NaN,NaN,0.867769,0.966907,0.650396,0.858440,0.899635,0.909451,0.754977,1607,265,55,493,0.879038,0.607846
4,mistral-small-3.1,NaN,NaN,0.876860,0.940649,0.697161,0.897436,0.806569,0.918535,0.747885,1680,192,106,442,0.852003,0.606110
5,gpt-oss-120b,NaN,NaN,0.897521,0.952116,0.740385,0.913462,0.843066,0.932388,0.788396,1710,162,86,462,0.878264,0.659730
6,gemma-3-27b-it,NaN,NaN,0.864050,0.957321,0.649386,0.862714,0.868613,0.907558,0.743169,1615,257,72,476,0.865663,0.593817
7,mistral-small-3.1,NaN,NaN,0.869421,0.935123,0.683544,0.893162,0.788321,0.913661,0.732203,1672,200,116,432,0.840742,0.586786
8,gpt-oss-120b,NaN,NaN,0.886364,0.941893,0.722675,0.909188,0.808394,0.925251,0.763135,1702,170,105,443,0.858791,0.627595
